## Notebook to test different QR implementations.

Note: `qr_pd` requires Spark 3.0

References:

- https://stackoverflow.com/questions/11144513/cartesian-product-of-x-and-y-array-points-into-single-array-of-2d-points

In [ ]:
from typing import List

import math
import numpy as np
import pandas as pd
from numba import njit
from pyspark.sql.functions import col, explode, lit, pandas_udf, rand, udf
from pyspark.sql.types import ArrayType, LongType

In [ ]:
@njit()
def qr_jit(xmin:float,
           ymin:float,
           xmax:float,
           ymax:float,
           cell:float
           ) -> List[int]:
    qmin = math.floor(xmin / cell)
    qmax = math.floor(xmax / cell) + 1
    rmin = math.floor(ymin / cell)
    rmax = math.floor(ymax / cell) + 1
    return [(q << 32 | r & 0xFFFFFFFF) for q in range(qmin, qmax) for r in range(rmin, rmax)]

In [ ]:
@udf(ArrayType(LongType()))
def qr(xmin:float,
       ymin:float,
       xmax:float,
       ymax:float,
       cell:float
      ) -> List[int]:
    return qr_jit(xmin, ymin, xmax, ymax, cell)

In [ ]:
@udf(ArrayType(LongType()))
def qr_np(xmin:float,
          ymin:float,
          xmax:float,
          ymax:float,
          cell:float
         ) -> List[int]:
    qmin = math.floor(xmin / cell)
    qmax = math.floor(xmax / cell) + 1
    rmin = math.floor(ymin / cell)
    rmax = math.floor(ymax / cell) + 1
    
    q = np.arange(qmin,qmax)
    r = np.arange(rmin,rmax)
    
    qr = np.dstack(np.meshgrid(q,r)).reshape(-1,2)
    
    return ((qr[:,0] << 31 ) | (qr[:,1] & 0xFFFFFFFF)).tolist()

In [ ]:
@njit()
def _qr(row):
    xmin,ymin,xmax,ymax,cell = row
    qmin = math.floor(xmin / cell)
    qmax = math.floor(xmax / cell) + 1
    rmin = math.floor(ymin / cell)
    rmax = math.floor(ymax / cell) + 1
    return [(q << 32 | r & 0xFFFFFFFF) for q in range(qmin, qmax) for r in range(rmin, rmax)]

@pandas_udf(ArrayType(LongType()))
def qr_pd(xmin: pd.Series,
          ymin: pd.Series,
          xmax: pd.Series,
          ymax: pd.Series,
          cell: pd.Series
          ) -> pd.Series:
    stack = np.dstack([xmin,ymin,xmax,ymax,cell]).reshape(-1,5)
    return pd.Series(map(_qr, stack))

In [ ]:
xmin,ymin,xmax,ymax = (-74.2555913638106, 40.496115395209344, -73.70000906387119, 40.91553277600007)

xdel = xmax - xmin
ydel = ymax - ymin

cell = 0.0002

In [ ]:
df = spark\
    .range(1_000_000)\
    .withColumn("xmin", xmin+xdel*rand())\
    .withColumn("ymin", ymin+ydel*rand())\
    .withColumn("xmax", col("xmin")+0.0001*rand())\
    .withColumn("ymax", col("ymin")+0.0001*rand())\
    .withColumn("cell", lit(cell))\
    .persist()

In [ ]:
df\
    .withColumn("qr", explode(qr("xmin","ymin","xmax","ymax","cell")))\
    .cache()\
    .createOrReplaceTempView("v0")

In [ ]:
%%time

sql("""
select max(pop) max_pop
from(
select qr,count(qr) pop
from v0
group by qr
)
""")\
    .show()

| CPU Method | Time | Count |
|--------|------|-------|
| qr | 11.9 s | 7 |
| qr_np | 20.9 s | 7 |
| qr_pd | 14.4 s | 7 |


| GPU Method | Time | Count |
|--------|------|-------|
| qr | 525 ms | 6 |
| qr_np | 530 ms | 6 |
| qr_pd | 553 ms | 6 |